# Conformal Prediction Quickstart

End-to-end **conformal prediction** for regression and classification using [crepes](https://github.com/henrikbostrom/crepes) and scikit-learn. The intervals and prediction sets are guaranteed to contain the true label with the user-specified coverage (e.g. 90%) under exchangeability — regardless of the underlying model.

## 1. Regression: 90% prediction intervals on California Housing

In [ ]:
import numpy as np
from crepes import ConformalRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

X, y = fetch_california_housing(return_X_y=True)
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_cal,  X_test, y_cal,  y_test  = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)

model = RandomForestRegressor(n_estimators=200, random_state=0).fit(X_train, y_train)
residuals_cal = np.abs(y_cal - model.predict(X_cal))

cr = ConformalRegressor().fit(residuals=residuals_cal)
y_hat = model.predict(X_test)
intervals = cr.predict(y_hat=y_hat, confidence=0.9)

lower, upper = intervals[:, 0], intervals[:, 1]
coverage = np.mean((y_test >= lower) & (y_test <= upper))
avg_width = np.mean(upper - lower)
print(f'Empirical coverage: {coverage:.3f}  (target = 0.90)')
print(f'Average interval width: {avg_width:.3f}')

## 2. Classification: conformal prediction sets on digits

In [ ]:
import numpy as np
from crepes import ConformalClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_cal,  X_test, y_cal,  y_test  = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)

clf = LogisticRegression(max_iter=2000).fit(X_train, y_train)

proba_cal = clf.predict_proba(X_cal)
alphas_cal = 1 - proba_cal[np.arange(len(y_cal)), y_cal]

cc = ConformalClassifier().fit(alphas=alphas_cal)
proba_test = clf.predict_proba(X_test)
alphas_test = 1 - proba_test
pred_sets = cc.predict_set(alphas=alphas_test, confidence=0.9)

set_sizes = pred_sets.sum(axis=1)
covered = pred_sets[np.arange(len(y_test)), y_test].mean()
print(f'Empirical coverage: {covered:.3f}  (target = 0.90)')
print(f'Average set size: {set_sizes.mean():.3f}')

## Next steps

See the main [README](../README.md) for 600+ more conformal prediction resources — including additional open-source libraries, papers, and tutorials.